# Phase 2, Part A: Diagnosing Data Quality Issues

Before applying any transformations, it is necessary to quantify the exact extent of the anomalies identified during the initial exploration. This approach ensures that all subsequent data cleaning decisions are strictly evidence based. The four specific anomalies to investigate are:

1. The `age` column contains a minimum value of 0, which is impossible for a credit applicant.
2. The `RevolvingUtilizationOfUnsecuredLines` column has a maximum value of 50,708, whereas utilisation ratios typically sit between 0 and 1.
3. The `DebtRatio` column exhibits a similar extreme outlier, reaching a maximum of 329,664.
4. The three delinquency columns (`NumberOfTime30-59DaysPastDueNotWorse`, `NumberOfTimes90DaysLate`, and `NumberOfTime60-89DaysPastDueNotWorse`) all reach a maximum value of exactly 98. It is highly improbable for three independent behavioural metrics to share the exact same extreme ceiling, suggesting a placeholder or system artefact rather than genuine data.


In [15]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(df.shape)  # should print (150000, 11) again - confirms the reload worked


(150000, 11)


## Issue 1: `age` of 0

An age of zero is physically impossible for a credit applicant. The following code identifies exactly how many records contain this error so they can be reviewed directly.

In [16]:
print("Rows with age == 0:", (df['age'] == 0).sum())
df[df['age'] == 0]


Rows with age == 0: 1


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
65696,0,1.0,0,1,0.436927,6000.0,6,0,2,0,2.0


**Resolution Strategy:** Because this anomaly affects only a single row out of 150,000, the most robust approach is to drop the record entirely. There is no reliable method to impute a borrower's actual age, and removing one row has zero statistical impact on the pipeline. It is important to note that the appropriate cleaning strategy depends heavily on the scale of the anomaly; if thousands of rows were affected, a completely different methodology would be required.

## Issue 2: `RevolvingUtilizationOfUnsecuredLines` Reaching 50,708

This column represents a ratio of currently used credit divided by total available credit. In reality, this metric should sit between 0 and 1. It is possible for it to marginally exceed 1 (for example, up to roughly 2) if fees or interest push a balance over the limit, but it cannot legitimately reach into the tens of thousands. 

The following code evaluates how many rows exceed various thresholds to determine the scale of these impossible values and inspects the most extreme outliers directly.


In [17]:
for threshold in [1, 2, 10, 100]:
    count = (df['RevolvingUtilizationOfUnsecuredLines'] > threshold).sum()
    print(f"Rows with utilisation > {threshold}: {count}")

print()
df['RevolvingUtilizationOfUnsecuredLines'].sort_values(ascending=False).head(10)


Rows with utilisation > 1: 3321
Rows with utilisation > 2: 371
Rows with utilisation > 10: 241
Rows with utilisation > 100: 223



85490     50708.0
31415     29110.0
16957     22198.0
149161    22000.0
149280    20514.0
117316    18300.0
21979     17441.0
124534    13930.0
72593     13498.0
71706     13400.0
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64

**Resolution Strategy:** The approach to this anomaly differs fundamentally from the zero age issue. A utilisation rate of 1.3, representing 130 per cent, is highly unusual but mechanically possible if a borrower has exceeded their credit limit. Conversely, a utilisation rate in the thousands is a definitive data entry error. This distinction is critical for robust risk modelling. Values that are extreme yet meaningful carry genuine predictive signal and should be retained and capped, whereas physically impossible values must be treated as errors. The cleaning logic applied in the subsequent phase will establish a strict threshold to neutralise the impossible tail without destroying the legitimate signal of overextended borrowers.

## Issue 3: `DebtRatio` Reaching 329,664 and the Link to Missing Income

It is important to test whether this column behaves differently for the approximately 20 per cent of records where `MonthlyIncome` is missing. A debt ratio is mechanically calculated as monthly debt obligations divided by monthly income. If the underlying income data was missing or corrupted upstream, this ratio calculation would break, resulting in extreme anomalies. Evaluating this hypothesis directly ensures the cleaning strategy addresses the root cause of the error rather than just the symptom.


In [18]:
print("DebtRatio where MonthlyIncome is MISSING:")
print(df[df['MonthlyIncome'].isnull()]['DebtRatio'].describe())

print()
print("DebtRatio where MonthlyIncome is PRESENT:")
print(df[df['MonthlyIncome'].notnull()]['DebtRatio'].describe())


DebtRatio where MonthlyIncome is MISSING:
count     29731.000000
mean       1673.396556
std        4248.372895
min           0.000000
25%         123.000000
50%        1159.000000
75%        2382.000000
max      329664.000000
Name: DebtRatio, dtype: float64

DebtRatio where MonthlyIncome is PRESENT:
count    120269.000000
mean         26.598777
std         424.446457
min           0.000000
25%           0.143388
50%           0.296023
75%           0.482559
max       61106.500000
Name: DebtRatio, dtype: float64


**Resolution Strategy:** Comparing the mean and maximum values between the two groups reveals that records with missing income contain drastically higher and more volatile `DebtRatio` figures. This provides concrete evidence that these data quality issues are directly linked. Consequently, missing income cannot simply be overwritten with a generic average. It must be preserved as a distinct, meaningful signal during the data cleaning process.


## Issue 4: The Suspicious '98' Ceiling Across Delinquency Columns

Three independent measures of late payments (30 to 59 days, 60 to 89 days, and 90 plus days) all reach a maximum value of exactly 98. It is highly improbable for three separate behavioural metrics to naturally hit the exact same extreme ceiling. This strongly suggests a shared placeholder or data encoding error rather than genuine payment history. 

The following code verifies whether the exact same rows are affected across all three columns to confirm the presence of a systemic data artefact.

In [19]:
delinquency_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]

for col in delinquency_cols:
    print(f"Rows with {col} >= 96:", (df[col] >= 96).sum())

print()
mask_suspect = df['NumberOfTime30-59DaysPastDueNotWorse'] >= 96
df[mask_suspect][delinquency_cols].head(10)


Rows with NumberOfTime30-59DaysPastDueNotWorse >= 96: 269
Rows with NumberOfTimes90DaysLate >= 96: 269
Rows with NumberOfTime60-89DaysPastDueNotWorse >= 96: 269



,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTimes90DaysLate,NumberOfTime60-89DaysPastDueNotWorse
1734,98,98,98
2287,98,98,98
3885,98,98,98
4418,98,98,98
4706,98,98,98
5074,98,98,98
6281,98,98,98
7033,98,98,98
7118,98,98,98
7688,98,98,98


## Diagnostic Conclusions and Next Steps

The diagnostic checks confirm the presence and scale of the four primary data quality issues. The zero age anomaly is isolated to a single record, while the extreme credit utilisation values contain a mix of plausible over limit behaviour and definitive errors. The '98' sentinel values align perfectly across all three delinquency columns, confirming a systemic placeholder. Furthermore, the analysis proves a direct mechanical link between missing income data and distorted debt ratios.

In the next section, these findings are translated into concrete data cleaning operations. The focus then shifts to feature engineering to construct the core affordability metrics: a robust debt to income measure, an explicit flag for missing or unreliable income, and a consolidated, cleaned delinquency history.

---
# Phase 2, Part B: Translating Diagnosis into Decisions

Based on the quantitative diagnostics completed in Part A, concrete cleaning and decision rules are established for each issue:

| Issue | Diagnostic Finding | Cleaning Decision |
|---|---|---|
| `age == 0` | Exactly 1 record affected | Drop the record entirely |
| Utilisation outliers | 3,321 rows exceed 1, but only 371 exceed 2 (reaching 50,708) | Cap utilisation at 2 |
| DebtRatio and missing income | Mean DebtRatio is 1,673 when income is missing versus 26.6 when present | Flag missingness, impute median income, and cap DebtRatio at 5 |
| '98' sentinel values | Affects the exact same rows across all three delinquency columns | Flag the error row, then cap each column at its own genuine maximum |

To ensure the notebook functions reliably when reopened on a different day with a fresh kernel, the raw dataset is reloaded fresh rather than relying on the workspace memory from Part A.

In [20]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(df.shape)


(150000, 11)


## Decision 1: Dropping the Single `age == 0` Row

Since the zero age anomaly is isolated to just one record, the row is removed from the dataset to maintain data integrity.

In [21]:
df = df[df['age'] > 0].copy()
print("New shape after dropping the age==0 row:", df.shape)


New shape after dropping the age==0 row: (149999, 11)


## Decision 2: Capping Utilisation at 2

Using `.clip(upper=2)` preserves any value at or below 2 while replacing any higher value with 2. This winsorization technique neutralizes the impossible tail values without deleting the records or destroying the valid signal for borrowers who are moderately over their credit limit.

In [22]:
UTILIZATION_CAP = 2
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=UTILIZATION_CAP)
df['RevolvingUtilizationOfUnsecuredLines'].describe()


count    149999.000000
mean          0.324485
std           0.364696
min           0.000000
25%           0.029867
50%           0.154176
75%           0.559044
max           2.000000
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64

## Decision 3: Income Missing Flag, Imputation, and DebtRatio Cap

The sequence of these operations is critical: the `income_missing` flag must be created before any imputation occurs. Otherwise, the information indicating which records originally lacked income data would be permanently lost.


In [23]:
# 1. Preserve the missingness signal as its own feature, before touching the column
df['income_missing'] = df['MonthlyIncome'].isnull().astype(int)
print(df['income_missing'].value_counts())

# 2. Impute missing income with the median of the KNOWN incomes only
median_income = df.loc[df['MonthlyIncome'].notnull(), 'MonthlyIncome'].median()
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(median_income)
print(f"\nImputed missing MonthlyIncome with the median: {median_income}")

# 3. Cap DebtRatio - a ratio of 5 already means monthly debt payments are five times
#    monthly income, already extreme distress. Beyond that we're almost certainly
#    looking at a broken upstream calculation rather than a real value.
DEBT_RATIO_CAP = 5
df['DebtRatio'] = df['DebtRatio'].clip(upper=DEBT_RATIO_CAP)
df['DebtRatio'].describe()


income_missing
0    120268
1     29731
Name: count, dtype: int64

Imputed missing MonthlyIncome with the median: 5400.0


count    149999.000000
mean          1.286969
std           1.884549
min           0.000000
25%           0.175074
50%           0.366503
75%           0.868257
max           5.000000
Name: DebtRatio, dtype: float64

## Handling Missing Values in `NumberOfDependents`

A missing data gap in `NumberOfDependents` was overlooked during the initial cleaning phase, leaving unhandled NaN values that caused downstream model fitting to fail. Logistic regression algorithms cannot process missing values, highlighting the necessity of rigorous data validation. 

**Resolution Strategy:** Following the same logic used for `MonthlyIncome`, an explicit binary flag is created first to preserve the missingness indicator. The remaining missing values in `NumberOfDependents` are then safely imputed using the median value from the training data, ensuring the dataset is fully complete before modelling.

In [24]:
df['dependents_missing'] = df['NumberOfDependents'].isnull().astype(int)
median_dependents = df.loc[df['NumberOfDependents'].notnull(), 'NumberOfDependents'].median()
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(median_dependents)

print(df['dependents_missing'].value_counts())
print(f"Imputed missing NumberOfDependents with the median: {median_dependents}")


dependents_missing
0    146075
1      3924
Name: count, dtype: int64
Imputed missing NumberOfDependents with the median: 0.0


## Decision 4: Flagging and Capping the 98 Sentinel Values

Each delinquency column is flagged prior to modification to preserve the information that a data corruption event occurred. Rather than using arbitrary thresholds, each column is capped at its own highest genuine non-sentinel value derived directly from the dataset.

In [25]:
delinquency_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]

df['has_delinquency_data_error'] = (df[delinquency_cols] >= 96).any(axis=1).astype(int)
print(df['has_delinquency_data_error'].value_counts())

for col in delinquency_cols:
    genuine_max = df.loc[df[col] < 96, col].max()
    df[col] = df[col].clip(upper=genuine_max)
    print(f"{col}: capped at {genuine_max}")


has_delinquency_data_error
0    149730
1       269
Name: count, dtype: int64
NumberOfTime30-59DaysPastDueNotWorse: capped at 13
NumberOfTimes90DaysLate: capped at 17
NumberOfTime60-89DaysPastDueNotWorse: capped at 11


## Feature Engineering Implementation

The engineered features are added to the dataset to capture financial sustainability and behavioural risk:

- `estimated_monthly_debt_payment` converts the `DebtRatio` into an actual monetary figure by taking the product of the debt ratio and `MonthlyIncome`.
- `disposable_income_estimate` represents the net monthly cash flow, calculated as `MonthlyIncome` minus `estimated_monthly_debt_payment`.
- `total_past_delinquencies` sums the cleaned values from `NumberOfTime30-59DaysPastDueNotWorse`, `NumberOfTime60-89DaysPastDueNotWorse`, and `NumberOfTimes90DaysLate` to quantify overall repayment distress.


In [26]:
df['estimated_monthly_debt_payment'] = df['DebtRatio'] * df['MonthlyIncome']
df['disposable_income_estimate'] = df['MonthlyIncome'] - df['estimated_monthly_debt_payment']
df['total_past_delinquencies'] = df[delinquency_cols].sum(axis=1)

df[['MonthlyIncome', 'DebtRatio', 'estimated_monthly_debt_payment',
    'disposable_income_estimate', 'total_past_delinquencies']].describe()


,MonthlyIncome,DebtRatio,estimated_monthly_debt_payment,disposable_income_estimate,total_past_delinquencies
count,1.499990e+05,149999.000000,149999.000000,1.499990e+05,149999.000000
mean,6.418458e+03,1.286969,6653.696404,-2.352387e+02,0.473876
std,1.289044e+04,1.884549,9905.122519,1.625516e+04,2.040392
min,0.000000e+00,0.000000,0.000000,-1.333320e+05,0.000000
25%,3.903000e+03,0.175074,766.168268,3.367832e+02,0.000000
50%,5.400000e+03,0.366503,2102.787165,2.837280e+03,0.000000
75%,7.400000e+03,0.868257,4797.447665,5.287121e+03,0.000000
max,3.008750e+06,5.000000,166665.000000,3.004327e+06,41.000000


## Evaluating Unsustainable Financial Profiles

Running an inspection on the newly engineered features reveals the precise count of borrowers operating with a negative disposable income. This metric serves as a powerful quantitative highlight for the project README, directly demonstrating the economic vulnerability captured by the model.

In [27]:
negative_count = (df['disposable_income_estimate'] < 0).sum()
print("Rows with negative estimated disposable income:", negative_count)
print(f"That's {100 * negative_count / len(df):.1f}% of the cleaned dataset")


Rows with negative estimated disposable income: 33616
That's 22.4% of the cleaned dataset


## Saving the Cleaned Dataset

The cleaned and feature engineered dataset is exported to a dedicated file. This decouples Phase 3 modelling from Phase 2 data cleaning, ensuring that the pipeline remains modular, reproducible, and easy to maintain.


In [28]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cleaned_features.csv')
print("Saved. Final shape:", df.shape)


Saved. Final shape: (149999, 17)


## Notebook Completion and Summary

Notebook 2 is now fully structured, moving the pipeline from raw anomaly diagnosis through rigorous data cleaning to robust feature engineering. The dataset is thoroughly prepped, free of physically impossible values, supplemented with explicit missingness flags, and enriched with intuitive affordability metrics like estimated monthly debt payments and disposable income. 

With a clean, modular output saved and ready, the project shifts directly into Phase 3 to train, evaluate, and interpret the classification models.
